# Eppendorf adapter top-and-center geometry test

This notebook picks up the **1000 µL filtered tip at A1** from the rack in **tip-carrier position 0**. The tip carrier is loaded at **rail 25**, and the 32-position tube carrier with Eppendorf inserts is loaded at **rail 19**.

Physical channel 1 (PyLabRobot channel 0) moves to the X/Y center of the frontmost Eppendorf adapter (tube-carrier site 0). It stops first 5 mm above the modeled adapter top, then lowers only to the adapter-top plane. **No tubes may be present; all Eppendorf adapters must be installed.** Keep the deck clear and supervise every move.

**Run order:** Execute cells from top to bottom once. Do not rerun the deck-layout cell after setup because it creates a new, uninitialized `LiquidHandler`.

2026-07-04
I verified the x, y offsets to put 1000ul filtered tip in center of tube carrier with adapter

In [1]:
%load_ext autoreload
%autoreload 2

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources import hamilton_96_tiprack_1000uL_filter
from pylabrobot.resources.hamilton import (
    STARLetDeck,
    TIP_CAR_480_A00,
    hamilton_tube_carrier_32_a00_insert_eppendorf_1_5mL,
)

In [2]:
# PyLabRobot channel and carrier positions are zero-based.
CHANNEL = 0  # physical channel 1
TIP_CARRIER_POSITION = 0
TIP_POSITION = "A1"
INSERT_SITE = 0  # frontmost Eppendorf insert
SAFE_TRAVERSE_Z = 247.6
ABOVE_TOP_CLEARANCE_MM = 5.0

backend = STARBackend()
lh = LiquidHandler(backend=backend, deck=STARLetDeck())

tip_carrier = TIP_CAR_480_A00(name="tip_carrier")
tip_rack_1000ul = hamilton_96_tiprack_1000uL_filter(
    name="tips_1000ul_filter"
)
tip_carrier[TIP_CARRIER_POSITION].assign_child_resource(tip_rack_1000ul)
lh.deck.assign_child_resource(tip_carrier, rails=25)

tube_carrier = hamilton_tube_carrier_32_a00_insert_eppendorf_1_5mL(
    name="tube_carrier"
)
lh.deck.assign_child_resource(tube_carrier, rails=19)

adapter_center = tube_carrier[INSERT_SITE].get_absolute_location(x="c", y="c")
ADAPTER_CENTER_X = adapter_center.x
ADAPTER_CENTER_Y = adapter_center.y
ADAPTER_TOP_Z = tube_carrier.get_absolute_location(z="t").z
ABOVE_ADAPTER_Z = ADAPTER_TOP_Z + ABOVE_TOP_CLEARANCE_MM

assert TIP_CARRIER_POSITION == 0
assert INSERT_SITE == min(tube_carrier.sites), "Site 0 must be frontmost"
assert tip_rack_1000ul[TIP_POSITION][0].has_tip(), (
    f"No tip modeled at {TIP_POSITION}"
)
selected_tip = tip_rack_1000ul[TIP_POSITION][0].get_tip()
assert selected_tip.nominal_volume == 1000
assert selected_tip.has_filter
assert ADAPTER_TOP_Z < ABOVE_ADAPTER_Z < SAFE_TRAVERSE_Z

print(lh.deck.summary())
print("\n=== ADAPTER GEOMETRY (absolute deck coordinates, mm) ===")
print(
    f"Modeled frontmost adapter center: "
    f"X={ADAPTER_CENTER_X:.2f}, Y={ADAPTER_CENTER_Y:.2f}"
)
print(f"Modeled adapter top:              Z={ADAPTER_TOP_Z:.2f}")
print(f"Above-adapter approach:           Z={ABOVE_ADAPTER_Z:.2f}")
print(f"Safe traverse:           Z={SAFE_TRAVERSE_Z:.2f}")
print("1000 µL tip source:      carrier position 0, A1")

Rail  Resource                      Type                 Coordinates (mm)
(-6)  ├── trash_core96              Trash                (-58.200, 106.000, 216.400)
      │
(19)  ├── tube_carrier              TubeCarrier          (505.000, 063.000, 100.000)
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │   ├── <empty>
      │
(25)  ├── tip_carrier               

## Connect and run

Confirm the printed layout matches the physical deck before setup. The 1000 µL filtered-tip rack must be in position 0 of the tip carrier at rail 25, with a tip at A1. The tube carrier must be at rail 19 with all Eppendorf adapters installed and no tubes present.

In [3]:
if lh.setup_finished:
    print("Liquid handler setup is already complete; setup was not repeated.")
else:
    await lh.setup(skip_autoload=True)

2026-07-04 10:43:11,828 - pylabrobot.io.usb - INFO - Finding USB device...
2026-07-04 10:43:11,838 - pylabrobot.io.usb - INFO - Found USB device.
2026-07-04 10:43:11,841 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-07-04 10:43:15,012 - pylabrobot - INFO - Running backend initialization procedure.


In [8]:
def confirm(message: str):
    response = input(f"\n{message} Type GO and press Enter to continue: " ).strip().upper()
    if response != "GO":
        raise RuntimeError("Operator cancelled the geometry test.")


async def run_adapter_top_center_test(
    x_offset_mm: float = 0.0,
    y_offset_mm: float = 0.0,
):
    target_x = ADAPTER_CENTER_X + x_offset_mm
    target_y = ADAPTER_CENTER_Y + y_offset_mm
    tip_is_attached = False

    print("\n=== ADJUSTED ADAPTER TARGET (mm) ===")
    print(f"X: {ADAPTER_CENTER_X:.2f} + ({x_offset_mm:+.2f}) = {target_x:.2f}")
    print(f"Y: {ADAPTER_CENTER_Y:.2f} + ({y_offset_mm:+.2f}) = {target_y:.2f}")
    print(f"Adapter top Z: {ADAPTER_TOP_Z:.2f}")

    confirm(
        "Verify: 1000 µL filtered tip at tip-carrier position 0, A1; "
        "all Eppendorf adapters installed at rail 19; NO TUBES present; "
        "deck clear."
    )
    await lh.pick_up_tips(
        tip_rack_1000ul[TIP_POSITION], use_channels=[CHANNEL]
    )
    tip_is_attached = True

    try:
        await lh.prepare_for_manual_channel_operation(CHANNEL)
        await lh.backend.move_channel_tool_z(
            channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z
        )

        confirm(
            f"Ready to move at safe Z to adjusted adapter-site 0 center "
            f"X={target_x:.2f}, Y={target_y:.2f}."
        )
        await lh.move_channel_x(CHANNEL, target_x)
        await lh.move_channel_y(CHANNEL, target_y)

        confirm(
            f"Ready to lower to Z={ABOVE_ADAPTER_Z:.2f}, "
            f"{ABOVE_TOP_CLEARANCE_MM:.1f} mm above the adapter top."
        )
        await lh.backend.move_channel_tool_z(
            channel_idx=CHANNEL, z=ABOVE_ADAPTER_Z
        )

        confirm(
            f"Verify the adjusted X/Y center. Ready to lower only to the "
            f"modeled adapter-top plane at Z={ADAPTER_TOP_Z:.2f}."
        )
        await lh.backend.move_channel_tool_z(
            channel_idx=CHANNEL, z=ADAPTER_TOP_Z
        )
        print("Tip is at the adjusted center and modeled top of adapter site 0.")

        confirm("Inspection complete; ready to raise and return the tip.")
        await lh.backend.move_channel_tool_z(
            channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z
        )
        await lh.drop_tips(
            tip_rack_1000ul[TIP_POSITION], use_channels=[CHANNEL]
        )
        tip_is_attached = False
        print(f"Adapter top-and-center test complete; tip returned to {TIP_POSITION}.")
    finally:
        if tip_is_attached:
            print("Test interrupted with a tip attached; attempting a safe raise.")
            try:
                await lh.backend.move_channel_tool_z(
                    channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z
                )
                print("Channel raised. Run the recovery cell to return the tip.")
            except Exception as recovery_error:
                print(f"Automatic raise failed: {recovery_error!r}")
                print("Recover the channel manually before moving deck hardware.")

In [17]:
# Edit these offsets for each test. Positive X/Y follows deck coordinates.
await run_adapter_top_center_test(x_offset_mm=-1.75, y_offset_mm=1.5)


=== ADJUSTED ADAPTER TARGET (mm) ===
X: 519.50 + (-1.75) = 517.75
Y: 77.50 + (+1.50) = 79.00
Adapter top Z: 160.20


Test interrupted with a tip attached; attempting a safe raise.
Channel raised. Run the recovery cell to return the tip.


RuntimeError: Operator cancelled the geometry test.

## Recovery

Run this only if the test was interrupted with the 1000 µL tip still attached.

In [18]:
await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)
await lh.drop_tips(tip_rack_1000ul[TIP_POSITION], use_channels=[CHANNEL])

## Stop

Run after the test or recovery is complete.

In [ ]:
await lh.stop()